# NB 33 — Packet P-015: Prediction Interval Calibration

**Agent OS v3.0 | Materia Arche**

## Decision Gate

> *If we say an 80% prediction interval covers [a, b], does 80% of actual T80 values fall within?*

Per-tree prediction intervals from our 700-tree Extra-Trees model give a
natural uncertainty estimate. For each test device, we collect all 700
individual tree predictions and use their distribution to form prediction
intervals at 50%, 80%, and 90% nominal coverage. This packet checks whether
those intervals are **calibrated**: do nominal coverage levels match empirical
coverage on held-out data?

| Item | Value |
|------|-------|
| Dataset | `perovskite_stability_clean.csv` (1,543 devices) |
| Target | `log1p(Stability_PCE_T80)` (trained), results in original hours |
| Model | Locked Extra-Trees (700 trees, sqrt features, seed 42) |
| Panel | Device 850 (T80=3400h), 1320 (T80=940h), 119 (T80=3423h) |
| Method | Per-tree prediction percentiles -> coverage check |

In [1]:
"""Cell 2 — Load data, train/test split, train locked ET, collect per-tree predictions."""
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split

# ── Load ──
df = pd.read_csv("perovskite_stability_clean.csv").fillna(0)
print(f"Dataset: {df.shape[0]} rows, {df.shape[1]} columns (NaN filled with 0)")

# ── Features & target ──
FEATURES = ['Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
            'MA', 'FA', 'Cs',
            'first_Prvskt_annealing_temperature',
            'first_Prvskt_thermal_annealing_time',
            'Perovskite_thickness', 'Cell_area_measured',
            'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF']
TARGET = 'Stability_PCE_T80'

X = df[FEATURES].copy()
y_log = np.log1p(df[TARGET].values)
y_hours = df[TARGET].values
print(f"Features: {len(FEATURES)},  Target: log1p({TARGET})")
print(f"y_log range: [{y_log.min():.2f}, {y_log.max():.2f}]")
print(f"y_hours range: [{y_hours.min():.1f}, {y_hours.max():.1f}]\n")

# ── 80/20 split ──
X_train, X_test, y_train_log, y_test_log, idx_train, idx_test = train_test_split(
    X, y_log, np.arange(len(X)), test_size=0.2, random_state=42
)
y_test_hours = np.expm1(y_test_log)
print(f"Train: {len(X_train)},  Test: {len(X_test)}")

# ── Train locked ET ──
ET_PARAMS = dict(
    n_estimators=700, max_features='sqrt', min_samples_split=3,
    min_samples_leaf=1, bootstrap=False, random_state=42, n_jobs=-1
)
et = ExtraTreesRegressor(**ET_PARAMS)
et.fit(X_train, y_train_log)
print("ExtraTrees trained on training set.\n")

# ── Collect per-tree predictions for every test device ──
# tree_preds shape: (n_test, 700)
tree_preds_log = np.array([tree.predict(X_test.values) for tree in et.estimators_]).T
print(f"Per-tree predictions collected: {tree_preds_log.shape} (devices x trees)")

# ── Compute prediction intervals at 50%, 80%, 90% coverage ──
COVERAGES = [50, 80, 90]
intervals = {}  # coverage -> (lower_log, upper_log)

for cov in COVERAGES:
    lo_pct = (100 - cov) / 2
    hi_pct = (100 + cov) / 2
    lower_log = np.percentile(tree_preds_log, lo_pct, axis=1)
    upper_log = np.percentile(tree_preds_log, hi_pct, axis=1)
    intervals[cov] = (lower_log, upper_log)
    lower_hours = np.expm1(lower_log)
    upper_hours = np.expm1(upper_log)
    print(f"\n{cov}% PI (hours): median width = {np.median(upper_hours - lower_hours):.1f} h")
    print(f"  Lower: median={np.median(lower_hours):.1f}, Upper: median={np.median(upper_hours):.1f}")

Dataset: 1543 rows, 47 columns (NaN filled with 0)
Features: 16,  Target: log1p(Stability_PCE_T80)
y_log range: [0.00, 9.04]
y_hours range: [0.0, 8400.0]

Train: 1234,  Test: 309


ExtraTrees trained on training set.

Per-tree predictions collected: (309, 700) (devices x trees)

50% PI (hours): median width = 237.8 h
  Lower: median=66.6, Upper: median=336.0

80% PI (hours): median width = 568.9 h
  Lower: median=20.0, Upper: median=619.7

90% PI (hours): median width = 794.8 h
  Lower: median=8.4, Upper: median=803.2


In [2]:
"""Cell 3 — Calibration check: empirical vs nominal coverage."""

print("=" * 70)
print("PREDICTION INTERVAL CALIBRATION TABLE")
print("=" * 70)
print(f"{'Nominal':>10s}  {'Empirical':>10s}  {'Cal. Error':>10s}  {'Direction':>10s}")
print("-" * 50)

cal_errors = {}
for cov in COVERAGES:
    lower_log, upper_log = intervals[cov]
    lower_hours = np.expm1(lower_log)
    upper_hours = np.expm1(upper_log)

    # Check: actual T80 falls within [lower, upper]?
    in_interval = (y_test_hours >= lower_hours) & (y_test_hours <= upper_hours)
    empirical = in_interval.mean() * 100
    cal_err = abs(empirical - cov)
    cal_errors[cov] = cal_err
    direction = "over-covers" if empirical > cov else "under-covers"

    print(f"{cov:>9.0f}%  {empirical:>9.1f}%  {cal_err:>9.1f}pp  {direction:>10s}")

max_cal_err = max(cal_errors.values())
print(f"\nMax calibration error: {max_cal_err:.1f} pp")
print(f"Mean calibration error: {np.mean(list(cal_errors.values())):.1f} pp")

PREDICTION INTERVAL CALIBRATION TABLE
   Nominal   Empirical  Cal. Error   Direction
--------------------------------------------------
       50%       42.7%        7.3pp  under-covers
       80%       66.0%       14.0pp  under-covers
       90%       76.1%       13.9pp  under-covers

Max calibration error: 14.0 pp
Mean calibration error: 11.7 pp


In [3]:
"""Cell 4 — Calibration by T80 range (80% coverage)."""

print("=" * 70)
print("CALIBRATION BY T80 RANGE (80% nominal coverage)")
print("=" * 70)

bins = [(0, 200), (200, 500), (500, 1000), (1000, 2000), (2000, np.inf)]
bin_labels = ["0-200", "200-500", "500-1000", "1000-2000", "2000+"]

lower_log_80, upper_log_80 = intervals[80]
lower_hours_80 = np.expm1(lower_log_80)
upper_hours_80 = np.expm1(upper_log_80)
in_80 = (y_test_hours >= lower_hours_80) & (y_test_hours <= upper_hours_80)

print(f"{'T80 Range':>12s}  {'N devices':>10s}  {'Empirical':>10s}  {'Cal. Error':>10s}")
print("-" * 50)

for (lo, hi), label in zip(bins, bin_labels):
    mask = (y_test_hours >= lo) & (y_test_hours < hi)
    n = mask.sum()
    if n > 0:
        emp = in_80[mask].mean() * 100
        err = abs(emp - 80)
        print(f"{label:>12s}  {n:>10d}  {emp:>9.1f}%  {err:>9.1f}pp")
    else:
        print(f"{label:>12s}  {n:>10d}  {'N/A':>10s}  {'N/A':>10s}")

print(f"\nTotal test devices: {len(y_test_hours)}")

CALIBRATION BY T80 RANGE (80% nominal coverage)
   T80 Range   N devices   Empirical  Cal. Error
--------------------------------------------------
       0-200         171       67.8%       12.2pp
     200-500          74       82.4%        2.4pp
    500-1000          42       54.8%       25.2pp
   1000-2000          15       20.0%       60.0pp
       2000+           7       14.3%       65.7pp

Total test devices: 309


In [4]:
"""Cell 5 — Panel device prediction intervals (80% PI vs actual T80)."""

PANEL_INDICES = [850, 1320, 119]
PANEL_T80 = {850: 3400, 1320: 940, 119: 3423}
PANEL_LABELS = {
    850:  "Device 850  (T80=3400h)",
    1320: "Device 1320 (T80=940h)",
    119:  "Device 119  (T80=3423h)"
}

print("=" * 70)
print("PANEL DEVICE 80% PREDICTION INTERVALS")
print("=" * 70)

# Check which panel devices are in test set vs train set
test_set = set(idx_test)
train_set = set(idx_train)

for dev_idx in PANEL_INDICES:
    label = PANEL_LABELS[dev_idx]
    actual_t80 = PANEL_T80[dev_idx]

    if dev_idx in test_set:
        # Device is in test set — use existing tree predictions
        pos = np.where(idx_test == dev_idx)[0][0]
        dev_tree_preds_log = tree_preds_log[pos, :]
        note = "(in test set)"
    else:
        # Leave-one-out: train on all except this device, predict it
        mask_loo = np.arange(len(X)) != dev_idx
        X_loo = X.iloc[mask_loo]
        y_loo = y_log[mask_loo]
        et_loo = ExtraTreesRegressor(**ET_PARAMS)
        et_loo.fit(X_loo, y_loo)
        dev_tree_preds_log = np.array([t.predict(X.iloc[[dev_idx]].values)[0]
                                        for t in et_loo.estimators_])
        note = "(leave-one-out)"

    # Compute 80% PI
    lo_log = np.percentile(dev_tree_preds_log, 10)
    hi_log = np.percentile(dev_tree_preds_log, 90)
    mean_log = dev_tree_preds_log.mean()

    lo_hours = np.expm1(lo_log)
    hi_hours = np.expm1(hi_log)
    mean_hours = np.expm1(mean_log)

    in_pi = lo_hours <= actual_t80 <= hi_hours

    print(f"\n{label}  {note}")
    print(f"  Mean prediction:  {mean_hours:>10.0f} h  (log: {mean_log:.3f})")
    print(f"  80% PI:           [{lo_hours:>8.0f}, {hi_hours:>8.0f}] h")
    print(f"  PI width:         {hi_hours - lo_hours:>10.0f} h")
    print(f"  Actual T80:       {actual_t80:>10.0f} h")
    print(f"  In 80% interval:  {'YES' if in_pi else 'NO'}")

PANEL DEVICE 80% PREDICTION INTERVALS



Device 850  (T80=3400h)  (leave-one-out)
  Mean prediction:        2257 h  (log: 7.722)
  80% PI:           [    1241,     3400] h
  PI width:               2159 h
  Actual T80:             3400 h
  In 80% interval:  NO

Device 1320 (T80=940h)  (in test set)
  Mean prediction:         419 h  (log: 6.041)
  80% PI:           [     113,      810] h
  PI width:                697 h
  Actual T80:              940 h
  In 80% interval:  NO



Device 119  (T80=3423h)  (leave-one-out)
  Mean prediction:         618 h  (log: 6.427)
  80% PI:           [     617,     1079] h
  PI width:                462 h
  Actual T80:             3423 h
  In 80% interval:  NO


In [5]:
"""Cell 6 — Interval width analysis."""

print("=" * 70)
print("INTERVAL WIDTH ANALYSIS (80% PI)")
print("=" * 70)

widths_hours = np.expm1(upper_log_80) - np.expm1(lower_log_80)
pred_mean_hours = np.expm1(tree_preds_log.mean(axis=1))
abs_errors = np.abs(y_test_hours - pred_mean_hours)

print(f"\n80% PI width statistics (hours):")
print(f"  Median width: {np.median(widths_hours):>10.1f} h")
print(f"  Mean width:   {np.mean(widths_hours):>10.1f} h")
print(f"  Min width:    {np.min(widths_hours):>10.1f} h")
print(f"  Max width:    {np.max(widths_hours):>10.1f} h")
print(f"  Std width:    {np.std(widths_hours):>10.1f} h")

# Correlation between interval width and absolute prediction error
corr = np.corrcoef(widths_hours, abs_errors)[0, 1]
print(f"\nCorrelation between 80% PI width and |prediction error|: {corr:.4f}")
if corr > 0.3:
    print("  -> Positive correlation: wider intervals tend to accompany larger errors.")
    print("     This is desirable — the model 'knows when it doesn't know'.")
elif corr > 0:
    print("  -> Weak positive correlation: some relationship between width and error.")
else:
    print("  -> No/negative correlation: interval width does not track prediction difficulty.")

# Width relative to prediction
rel_width = widths_hours / (pred_mean_hours + 1)  # +1 to avoid division by zero
print(f"\nRelative width (width / predicted T80):")
print(f"  Median: {np.median(rel_width):.2f}")
print(f"  Mean:   {np.mean(rel_width):.2f}")

INTERVAL WIDTH ANALYSIS (80% PI)

80% PI width statistics (hours):
  Median width:      568.9 h
  Mean width:        652.4 h
  Min width:           0.0 h
  Max width:        5392.8 h
  Std width:         585.0 h

Correlation between 80% PI width and |prediction error|: 0.3471
  -> Positive correlation: wider intervals tend to accompany larger errors.
     This is desirable — the model 'knows when it doesn't know'.

Relative width (width / predicted T80):
  Median: 4.22
  Mean:   5.27


In [6]:
"""Cell 7 — Save Packet_P015_Interval_Calibration.csv."""

# Compute all percentiles for test devices
pred_mean_log = tree_preds_log.mean(axis=1)
pred_p05 = np.percentile(tree_preds_log, 5, axis=1)
pred_p10 = np.percentile(tree_preds_log, 10, axis=1)
pred_p25 = np.percentile(tree_preds_log, 25, axis=1)
pred_p75 = np.percentile(tree_preds_log, 75, axis=1)
pred_p90 = np.percentile(tree_preds_log, 90, axis=1)
pred_p95 = np.percentile(tree_preds_log, 95, axis=1)

# Convert all to hours
out_df = pd.DataFrame({
    'device_idx': idx_test,
    'actual_T80': y_test_hours,
    'pred_mean': np.expm1(pred_mean_log),
    'pred_p10': np.expm1(pred_p10),
    'pred_p90': np.expm1(pred_p90),
    'pred_p05': np.expm1(pred_p05),
    'pred_p95': np.expm1(pred_p95),
    'pred_p25': np.expm1(pred_p25),
    'pred_p75': np.expm1(pred_p75),
    'in_80pct_interval': ((y_test_hours >= np.expm1(pred_p10)) &
                           (y_test_hours <= np.expm1(pred_p90))).astype(int)
})

out_path = "Packet_P015_Interval_Calibration.csv"
out_df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"  Rows: {len(out_df)} (test devices)")
print(f"  Columns: {list(out_df.columns)}")
print(f"\nPreview (first 10 rows):")
print(out_df.head(10).to_string(index=False, float_format='%.1f'))

Saved: Packet_P015_Interval_Calibration.csv
  Rows: 309 (test devices)
  Columns: ['device_idx', 'actual_T80', 'pred_mean', 'pred_p10', 'pred_p90', 'pred_p05', 'pred_p95', 'pred_p25', 'pred_p75', 'in_80pct_interval']

Preview (first 10 rows):
 device_idx  actual_T80  pred_mean  pred_p10  pred_p90  pred_p05  pred_p95  pred_p25  pred_p75  in_80pct_interval
       1493       180.0      173.7      44.3     605.4      15.8     827.5     101.8     396.1                  1
       1156         9.0       68.5       5.0     336.0       0.3     600.0      25.0     192.0                  1
       1253       120.0       94.6      14.8     774.6       2.3    1000.0      38.2     304.9                  1
        561        12.8       10.5      10.5      10.5      10.5      10.5      10.5      10.5                  0
       1098       252.0      184.0      31.1     600.0      13.7     710.2      89.4     386.7                  1
        582       359.5      300.1     340.6     383.3      69.6     383.

In [7]:
"""Cell 8 — Honest status footer."""

print("=" * 70)
print("PACKET P-015 — PREDICTION INTERVAL CALIBRATION")
print("=" * 70)

print(f"\nCalibration summary:")
for cov in COVERAGES:
    print(f"  {cov}% nominal -> cal. error = {cal_errors[cov]:.1f} pp")

max_cal_err = max(cal_errors.values())
print(f"\n  Max calibration error: {max_cal_err:.1f} pp")

# ── Verdict ──
print(f"\n{'─' * 70}")

if max_cal_err < 5:
    STATUS = "CONFIRMED"
    print(f"STATUS: *** {STATUS} ***")
    print(f"Prediction intervals are well-calibrated (max error < 5 pp).")
    print(f"The per-tree percentile intervals provide reliable uncertainty estimates.")
elif max_cal_err < 10:
    STATUS = "PROMISING"
    print(f"STATUS: *** {STATUS} ***")
    print(f"Prediction intervals are approximately calibrated (max error < 10 pp).")
    print(f"Useful for rough uncertainty quantification but not perfectly reliable.")
else:
    STATUS = "NEGATIVE"
    print(f"STATUS: *** {STATUS} ***")
    print(f"Prediction intervals are poorly calibrated (max error >= 10 pp).")
    print(f"Per-tree percentiles do NOT provide reliable coverage guarantees.")
    print(f"Conformal calibration or other post-hoc correction would be needed.")

print(f"\nLimitations:")
print(f"  - ExtraTrees with bootstrap=False: trees see identical data,")
print(f"    so tree-to-tree variation comes only from random split selection.")
print(f"  - Intervals are based on empirical tree prediction distribution,")
print(f"    not a proper probabilistic model.")
print(f"  - Calibration checked on a single 80/20 split (not cross-validated).")
print(f"{'─' * 70}")
print(f"\nPacket P-015 complete.  Status: {STATUS}")

PACKET P-015 — PREDICTION INTERVAL CALIBRATION

Calibration summary:
  50% nominal -> cal. error = 7.3 pp
  80% nominal -> cal. error = 14.0 pp
  90% nominal -> cal. error = 13.9 pp

  Max calibration error: 14.0 pp

──────────────────────────────────────────────────────────────────────
STATUS: *** NEGATIVE ***
Prediction intervals are poorly calibrated (max error >= 10 pp).
Per-tree percentiles do NOT provide reliable coverage guarantees.
Conformal calibration or other post-hoc correction would be needed.

Limitations:
  - ExtraTrees with bootstrap=False: trees see identical data,
    so tree-to-tree variation comes only from random split selection.
  - Intervals are based on empirical tree prediction distribution,
    not a proper probabilistic model.
  - Calibration checked on a single 80/20 split (not cross-validated).
──────────────────────────────────────────────────────────────────────

Packet P-015 complete.  Status: NEGATIVE
